# 06 pycavi SWarp

This notebooks run SWarp over the images for which Scamp creates a .head file. <br>
Tested with SWarp 2.38.0.

Uses the default configuration for SWarp, in case of needs, additional configuration parameters can be changed to optimize the performance in Chapman for example

In [1]:
import os
import sys
import glob
import logging
from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production

from pycavi import pycavi_tools
from pycavi.pycavi_astromatic_tools import get_swarp_lists_dict, prepare_to_swarp, run_swarp

This will be executed when the package is imported.


In [2]:
# set the directory folders
swarp_path = '/usr/bin/SWarp'
config_dir = os.getcwd() + '/' + '../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
swarp_dir_name = config['Directories']['swarp_dir_name'] 
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# all this part is to select all the .head files and grouping accordling to observation and filter
# the grouped images are stored in the swarp_list dictionary, where the key correspond to the combination
# of observation plus filter, and the element to corresponding list of images
# print(glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.head"))
#
# To select all data partitions leave the list below empty
#filters = set(['BB_B_99_ESO842', 'BB_V_89_ESO843', 'g_SDSS', 'r_SDSS', 'NB_659'])
filters = []

# We clean the list of bad observations - this should be done by hand
reject_observations=[]


# We first list the standard stars fields data partitions
swarp_lists_std = get_swarp_lists_dict(reduced_std_dir, reject_observations, filters)
std_observation_tags = list(set(swarp_lists_std.keys()))
std_observation_tags.sort()                # Needed to ensure same index in every run

# We next list the science observation fields data partitions
swarp_lists_sci = get_swarp_lists_dict(reduced_science_dir, reject_observations, filters)
sci_observation_tags = list(set(swarp_lists_sci.keys()))    
sci_observation_tags.sort()                # Needed to ensure same index in every run

reduced_dir_list = [reduced_science_dir]*len(sci_observation_tags) + [reduced_std_dir]*len(std_observation_tags)

obs_tags = sci_observation_tags + std_observation_tags
swarp_lists = swarp_lists_sci | swarp_lists_std

# We can reduce the selection by setting the list
# of indices to choose. Leave it empty to choose all.
#
idx_selected = []

for i in range(len(obs_tags)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    print("index: {} -- reduced_dir: {} / partition tag: {} -- nfiles: {} ".format(i, reduced_dir_list[i], obs_tags[i], len(swarp_lists[obs_tags[i]])))

    targets_to_swarp = swarp_lists[obs_tags[i]]

    counter = 0
    for target in targets_to_swarp:
        print("  ----------> ", os.path.basename(target))

index: 0 -- reduced_dir: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/ / partition tag: CP_Pup/NB_659 -- nfiles: 10 
  ---------->  OMEGA.2017-03-13T02:09:23.471_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:02:54.140_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:32:06.454_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:06:08.281_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:28:51.454_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:25:36.543_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:12:38.452_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:22:22.303_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:19:07.303_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:15:53.282_pre_wcs.head
index: 1 -- reduced_dir: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/ / partition tag: CP_Pup/r_SDSS -- nfiles: 10 
  ---------->  OMEGA.2017-03-13T02:38:02.286_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:39:07.286_pre_wcs.head
  ---------->  OMEGA.2017-03-13T02:36:57.286_pre

In [4]:
# We first select what to swarp
# If only a few data partitions write the indices to them below.
# The loop below will skip every index but those listed.
# If the list idx_selected is empty, then all data partitions will be swarped.
#

idx_selected = [1]

#
# updates some SWarp parameters 
# ---------- General processing parameters -----------------------
special_pars = {}

# coordinate type
special_pars["CELESTIAL_TYPE"] = "NATIVE"    #  "NATIVE, PIXEL, EQUATORIAL,GALACTIC,ECLIPTIC, or SUPERGALACTIC"
special_pars["PROJECTION_TYPE"] = "TAN"      # Any WCS projection code or NONE
                                             # TAN: Distorted tangential (default)
                                             # ------- Flux preserving that use PVi_j coeffs (See Table 2 of Swarp manual)
                                             # CEA: Cylindrical equal-area
                                             # COE: Conic equal area
                                             # BON: Bonne’s equal-area
                                             # and many others


special_pars["PROJECTION_ERR"]  = 0.001      # Maximum projection error (in output pixels), or 0 for no approximation

# processing parameters
special_pars["SATLEV_KEYWORD"] = "SATLEVEL"  # FITS key containing saturation level
special_pars["RESAMPLE_DIR"] = "./resamp"
special_pars["VMEM_DIR"] = "./swap"

# keywords to copy
PHASE3_KEYS = ",PRODCATG,PROG_ID,OBID1,NCOMBINE,OBSTECH,MJD-OBS,MJD-END,TIMESYS,FLUXCAL,RADESYS"
special_pars["COPY_KEYWORDS"] = "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"+PHASE3_KEYS


# ----------- Parameters governing sky subtraction ----------------
special_pars["BACK_SIZE"] = 128            # Background mesh size (pixels) all or for each image [128]
special_pars["BACK_FILTERSIZE"] = 3        # Background map filter range (meshes) (all or for each image) [3]
special_pars["BACK_FILTTHRESH"] = 0.0      # Threshold above which the background-map filter operates
special_pars["SUBTRACT_BACK"] = "Y"        # Allows background subtraction

# ------------Parameters governing Co-addition --------------------------
# To combine in one go choose the values below. This will remove individual files.
#
special_pars["COMBINE"] = "Y"              # Combine resampled images (Y/N)? If "N" automatically sets 
                                           # "DELETE_TMPFILES" to "N"
special_pars["RESAMPLE"] = "Y"             # In a second pass "RESAMPLE" should be set to "N", just to combine
                                           # frames. According to manual this works only if SAME_CRVAL and CDELT
                                           # which sounds a bit outdated so this should be investigated (no CDELT anymore).
special_pars["COMBINE_TYPE"] = "MEDIAN"    # "MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CHI2 or SUM

# We could also keep the individual swarped files by setting the parameter below to "N"
special_pars["DELETE_TMPFILES"] = "Y"

for i in range(len(obs_tags)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    observation_tag = obs_tags[i]
    logging.info("SWarping observation {}, filter {}".format(observation_tag.split("/")[0],
                                                             observation_tag.split("/")[1]))
    #print(swarp_lists[observation_tag])

    observation = swarp_lists[observation_tag][0].split("/")[-5]
    filter_ = swarp_lists[observation_tag][0].split("/")[-4]
    frames_list_name = observation + "_" + filter_ + ".list"
    special_pars["IMAGEOUT_NAME"] = frames_list_name[:-5] + ".coadd.fits"
    special_pars["WEIGHTOUT_NAME"] = frames_list_name[:-5] + ".weight.fits"    
    if "NB_659" in filter_:
        special_pars["BACK_SIZE"] = 2048
    swarp_dir_path = reduced_dir_list[i] + observation + '/' + swarp_dir_name

    # For debugging you might want to edit the frame list and run with 
    # reuse_frame_list = True.
    reuse_frame_list = True
    if frames_list_name is not None:
        if reuse_frame_list and os.path.exists(swarp_dir_path+'/'+frames_list_name):
            prepare_to_swarp(swarp_lists[observation_tag], swarp_dir_path, None, special_pars)
            run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars, path=swarp_path)
        else:
            frames_list_name = prepare_to_swarp(swarp_lists[observation_tag], swarp_dir_path, frames_list_name, special_pars)
            run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars, path=swarp_path)
    else:
        logging.info("Could not run swarp: frame_list_name is None")
    
logging.info('-------------- STD SWARP DONE ---------------') 

2024-10-30 14:49:15,560:INFO:SWarping observation CP_Pup, filter r_SDSS
2024-10-30 14:49:15,561:INFO:filter_: r_SDSS
2024-10-30 14:49:15,561:INFO:
          - observation: CP_Pup
          - filter_: r_SDSS
          - swarp_dir: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir
          - first_image_path: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre/OMEGA.2017-03-13T02:38:02.286_pre_wcs.head
2024-10-30 14:49:15,562:INFO:Attempting to create the SWarp folder..
2024-10-30 14:49:15,562:INFO:Folder already exists
2024-10-30 14:49:15,563:INFO:Creating frame list to swarp in directory: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/Swarp_dir
2024-10-30 14:49:15,563:INFO:swarp.config[CELESTIAL_TYPE] = NATIVE
2024-10-30 14:49:15,564:INFO:swarp.config[PROJECTION_TYPE] = TAN
2024-10-30 14:49:15,564:INFO:swarp.config[PROJECTION_ERR] = 0.001
2024-10-30 14:49:15,564:INFO:swarp.con

first_image_path: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre/OMEGA.2017-03-13T02:38:02.286_pre_wcs.head
run_command: cmdfile: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir/swarp_command.txt
run_command: logfile: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir/SWarp_logfile.log
run_command: str_cmd: /usr/bin/SWarp -c SWARP.conf  @CP_Pup_r_SDSS.list
run_command: cwd: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/Swarp_dir


run_command: cmdfile: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir/swarp_command.txt
run_command: str_cmd writting to command file /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir/swarp_command.txt
run_command: str_cmd written to command file /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Swarp_dir/swarp_command.txt
run_command: Running command: /usr/bin/

2024-10-30 14:55:13,684:INFO:
run_swarp: updating headers of image: CP_Pup_r_SDSS.coadd.fits
with headers from /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre/OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:55:13,684:INFO:reference_image: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre/OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:55:13,689:INFO:-------------- STD SWARP DONE ---------------
